# ALM frictionless components mortar contact condition (vector multiplier)

This notebook generates `custom_conditions/ALM_frictionless_components_mortar_contact_condition.cpp`, the
local left- and right-hand sides of `AugmentedLagrangianMethodFrictionlessComponentsMortarContactCondition<TDim, TNumNodes, TNormalVariation, TNumNodesMaster>`,
the **augmented Lagrangian frictionless** mortar contact condition with a **vector** Lagrange multiplier
(`VECTOR_LAGRANGE_MULTIPLIER`, `TDim` DoFs per slave node). Theory: thesis §4.3.3.2.2 and the
[Frictionless contact](https://kratosmultiphysics.github.io/Kratos/pages/Applications/Contact_Structural_Mechanics_Application/Theory/Frictionless_Contact.html)
page of the documentation.

## Formulation

### Components augmented Lagrangian (thesis eqs. 4.16-4.17)

The augmented multiplier is a vector, $\bar{\boldsymbol{\lambda}} = k\,\boldsymbol{\lambda} + \varepsilon\, \mathbf{n}\, g_n$, with
$\bar{\lambda}_n = k\,(\mathbf{n}\cdot\boldsymbol{\lambda}) + \varepsilon\, g_n$, and the weak form reads (thesis eq. 4.17b)

$$\delta\mathcal{L}_{co} = \int_{\Gamma_c^{(1)}} \begin{cases} \bar{\boldsymbol{\lambda}} \cdot \left( \delta\mathbf{u}^{(1)} - \delta\mathbf{u}^{(2)} \right) + k\, g_n\, \delta\boldsymbol{\lambda} \cdot \mathbf{n} & \bar{\lambda}_n \le 0 \quad \text{(contact zone)} \\[4pt] -\dfrac{k^2}{\varepsilon} \boldsymbol{\lambda} \cdot \delta\boldsymbol{\lambda} & \bar{\lambda}_n > 0 \quad \text{(gap zone)} \end{cases} \mathrm{d}\Gamma$$

while the multiplier equation also has to kill the tangential components (thesis eq. 4.17c):

$$\delta\mathcal{L}_{\lambda} = \int_{\Gamma_c^{(1)}} k \left( \mathbf{n} \cdot \delta\boldsymbol{\lambda} \right) g_n - \frac{k^2}{\varepsilon} \left( \boldsymbol{\lambda} - \mathbf{n} \left( \mathbf{n} \cdot \boldsymbol{\lambda} \right) \right) \cdot \left( \delta\boldsymbol{\lambda} - \mathbf{n} \left( \mathbf{n} \cdot \delta\boldsymbol{\lambda} \right) \right) \mathrm{d}\Gamma$$

The tangential multiplier is thus *penalised to zero* with the same $-k^2/\varepsilon$ coefficient used for
inactive nodes. Because the multiplier columns of the algebraic system (thesis eq. 4.36) are the full
$k\mathbf{D}^T$ blocks (diagonal with dual multipliers), this formulation admits the static condensation of the
multiplier block performed by `MixedULMLinearSolver`.

### Discrete form

With $\tilde{g}_{n,j} = -\mathbf{n}_j \cdot (\mathbf{D}\mathbf{x}^{(1)} - \mathbf{M}\mathbf{x}^{(2)})_j$, $\lambda_{n,j} = \mathbf{n}_j \cdot \boldsymbol{\lambda}_j$
and $\boldsymbol{\lambda}_{\tau,j} = \boldsymbol{\lambda}_j - \lambda_{n,j}\,\mathbf{n}_j$, the multiplier residuals of thesis eq. 4.36 are

$$\mathbf{r}_{\lambda_\mathcal{A}} = k\, \mathbf{n} \left( -\mathbf{n} \cdot ( \mathbf{D}\mathbf{x}_1 - \mathbf{M}\mathbf{x}_2 ) \right) - \frac{k^2}{\varepsilon} \boldsymbol{\tau} \cdot \boldsymbol{\lambda}, \qquad \mathbf{r}_{\lambda_\mathcal{I}} = \frac{k^2}{\varepsilon} \boldsymbol{\lambda}$$

### The two generated branches

Per slave node $j$, with $\mathcal{D}_j$ the `DynamicFactor` and $\bar{\boldsymbol{\lambda}}_j = k\boldsymbol{\lambda}_j + \varepsilon_j \tilde{g}_{n,j} \mathbf{n}_j$:

| branch | $\mathcal{R}_j$ |
|---|---|
| active | $\mathcal{D}_j\, \bar{\boldsymbol{\lambda}}_j \cdot \left( \mathbf{D}\,\mathbf{w}^{(1)} - \mathbf{M}\,\mathbf{w}^{(2)} \right)_j + k\, \tilde{g}_{n,j}\, \delta\lambda_{n,j} - \dfrac{k^2}{\varepsilon_j}\, \delta\boldsymbol{\lambda}_{\tau,j} \cdot \boldsymbol{\lambda}_{\tau,j}$ |
| inactive | $-\dfrac{k^2}{\varepsilon_j}\, \delta\boldsymbol{\lambda}_j \cdot \boldsymbol{\lambda}_j$ |

The run-time dispatch is, per node, `if (r_geometry[i].IsNot(ACTIVE)) {...} else {...}`.

## From the functional to the generated C++

All the mechanics of the generation live in `../mortar_condition_generator.py` (see also the
[Automatic differentiation](https://kratosmultiphysics.github.io/Kratos/pages/Applications/Contact_Structural_Mechanics_Application/Theory/Automatic_Differentiation.html)
page, thesis Appendix C):

1. **Symbols** (`SymbolSet`): the nodal unknowns `u1`, `u2` (displacements of the slave and master nodes),
   the multipliers, the test functions `w1`, `w2`, `wLM`, the reference coordinates `X1`, `X2`, the nodal
   normals `NormalSlave`, the mortar operators `DOperator`, `MOperator` and the parameters. The current
   coordinates are $\mathbf{x}^{(i)} = \mathbf{X}^{(i)} + \mathbf{u}^{(i)}$ and the nodal **weighted gap**
   (thesis eq. 4.31) is
   $$\tilde{g}_{n,j} = -\,\mathbf{n}_j \cdot \left( \mathbf{D}\, \mathbf{x}^{(1)} - \mathbf{M}\, \mathbf{x}^{(2)} \right)_j$$
   which is **positive for an open gap** and negative for penetration (`NormalGap` / `WEIGHTED_GAP`).
2. **AD exceptions** (thesis §C.3.1): $\mathbf{D}$, $\mathbf{M}$ and, when `TNormalVariation` is `true`, $\mathbf{n}$
   are not expressed in terms of the displacements. They are declared *undefined functions of the DoFs*
   (`DefineDofDependencyMatrix`), so that the chain rule produces unevaluated derivatives that are mapped to
   the arrays computed at run time by `DerivativesUtilities`:

   | symbolic node | C++ |
   |---|---|
   | `DOperator_i_j(u...)` | `DOperator(i,j)` |
   | `Derivative(DOperator_i_j(u...), u_k)` | `DeltaDOperator[k](i,j)` |
   | `Derivative(MOperator_i_j(u...), u_k)` | `DeltaMOperator[k](i,j)` |
   | `Derivative(NormalSlave_i_j(u...), u_k)` | `DeltaNormalSlave[k](i,j)` (normal variation only) |

   The index `k` runs over the slave displacement DoFs first and then the master ones, the ordering used by
   `MortarOperatorWithDerivatives`.
3. **Differentiation**: for every slave node $j$ and every active-set branch the functional $\mathcal{R}_j$
   returned by the function below is differentiated: $\mathbf{r} = \partial \mathcal{R} / \partial \mathbf{w}$
   (local RHS) and $\mathbf{K} = -\partial \mathbf{r} / \partial \mathbf{d}$ (local LHS), with the DoF vector
   $\mathbf{d} = [\mathbf{u}^{(2)}, \mathbf{u}^{(1)}, \boldsymbol{\lambda}]$ ordered *master, slave, multiplier* exactly as
   `GetDofList`. This is the Kratos convention $\mathbf{K}\,\Delta\mathbf{d} = \mathbf{r}$.
4. **Printing**: the derivative nodes are replaced by plain symbols, `sympy.cse` collects the common factors
   (`clhs*`, `crhs*`) and `sympy.ccode` prints C++; only the non-zero entries are emitted, accumulated with `+=`.
5. **Assembly of the file**: one `CalculateLocalLHS` specialisation per geometry pair (`2D2N`, `3D3N`, `3D4N`,
   `3D3N4N`, `3D4N3N`) and per `TNormalVariation` value; the RHS does **not** depend on the derivatives of the
   normal, so `StaticCalculateLocalRHS` is generated only for `TNormalVariation = false` and the `true`
   specialisation forwards to it. The bodies are substituted into the `*_template.cpp` of this folder at the
   `// replace_lhs` / `// replace_rhs` markers and the result is written once.

**Sign convention of the test-function quantities.** Every `<quantity>w` symbol (`NormalwGap`, `TangentwSlip*`)
is *minus* the variation of the quantity in the direction of the test functions, $X_w = -\delta X$: with the
gap defined as above, `NormalwGap = +n.(D w1 - M w2)`, so that the virtual work of a traction $\mathbf{t}$
is written $\mathbf{t} \cdot X_w$ and the residual is $-\delta\Pi$ (the force acting on the bodies).

## How to run this notebook

* **Requirements**: Python 3 and `sympy` (any modern version, tested with 1.14). A compiled Kratos is *not*
  needed: the shared module `../mortar_condition_generator.py` imports `custom_sympy_fe_utilities.py` and the
  core `sympy_fe_utilities.py` directly from the source tree.
* **Interactively**: open it with Jupyter from this folder and run all cells.
* **Headless** (no Jupyter installed): `python3 ../run_notebook.py <this notebook>` executes the code cells
  with the standard library only.
* The equivalent command-line script `generate_*.py` in this folder contains the *same* functional and
  generation call; keep both in sync when the formulation changes.

The output is written directly into `custom_conditions/` (overwriting the committed file). The generation of
the five geometries and the two normal-variation flags takes from a few minutes (frictionless) to about an
hour (ALM frictional). Restrict `COMBINATIONS` / `NORMAL_VARIATIONS` in the configuration cell for a quick
test, or set `OUTPUT_DIR` to a scratch folder.

In [ ]:
import os
import sys

# The shared generator module lives one folder up (automatic_differentiation/)
sys.path.insert(0, os.path.abspath(".."))
import mortar_condition_generator as generator

print("sympy", generator.sympy.__version__)

## Symbols of `SymbolSet` used by the functional

| symbol | meaning | C++ counterpart |
|---|---|---|
| `s.LM`, `s.LMNormal[j]`, `s.LMTangent.row(j)` | $\boldsymbol{\lambda}_j$, $\lambda_{n,j}$, $\boldsymbol{\lambda}_{\tau,j}$ | `VECTOR_LAGRANGE_MULTIPLIER` |
| `s.wLM`, `s.wLMNormal[j]`, `s.wLMTangent.row(j)` | test multiplier and its components | differentiated away |
| `s.NormalGap[j]` | $\tilde{g}_{n,j}$ | `WEIGHTED_GAP` |
| `s.NormalSlave.row(j)` | $\mathbf{n}_j$ | `NORMAL` |
| `s.Dw1Mw2.row(j)` | $(\mathbf{D}\,\mathbf{w}^{(1)} - \mathbf{M}\,\mathbf{w}^{(2)})_j$ | — |
| `s.ScaleFactor`, `s.PenaltyParameter[j]`, `s.DynamicFactor[j]` | $k$, $\varepsilon_j$, $\mathcal{D}_j$ | `SCALE_FACTOR`, `INITIAL_PENALTY`, `DYNAMIC_FACTOR` |

The branch identifiers passed to the functional are `inactive` and `active`.

## The functional

This is the physics of the condition; it is the only family-specific input of the generator.

In [ ]:
def frictionless_components_functional(s, node, branch):
    """Galerkin functional of one slave node (thesis eqs. 4.36-4.44 with a vector multiplier whose
    tangential part is penalised away). ``branch`` is ``inactive`` or ``active``."""
    rv_galerkin = 0
    if branch == "active":
        augmented_lm = (s.ScaleFactor * s.LM.row(node) + s.PenaltyParameter[node] * s.NormalGap[node] * s.NormalSlave.row(node))
        rv_galerkin += s.DynamicFactor[node] * (augmented_lm).dot(s.Dw1Mw2.row(node))
        rv_galerkin += s.ScaleFactor * s.NormalGap[node] * s.wLMNormal[node]
        rv_galerkin -= s.ScaleFactor**2 / s.PenaltyParameter[node] * (s.wLMTangent.row(node).dot(s.LMTangent.row(node)))
    else:
        rv_galerkin -= s.ScaleFactor**2 / s.PenaltyParameter[node] * (s.wLM.row(node).dot(s.LM.row(node)))
    return rv_galerkin

## Configuration

`COMBINATIONS` holds the `(dimension, slave nodes, master nodes)` triplets to generate and
`NORMAL_VARIATIONS` the values of `TNormalVariation`. The output goes to `custom_conditions/` by default.

In [ ]:
COMBINATIONS = generator.DEFAULT_COMBINATIONS   # ((2, 2, 2), (3, 3, 3), (3, 4, 4), (3, 3, 4), (3, 4, 3))
NORMAL_VARIATIONS = (False, True)
TEMPLATE_DIR, OUTPUT_DIR = generator.DefaultDirectories(os.getcwd())
print("template folder:", TEMPLATE_DIR)
print("output folder  :", OUTPUT_DIR)

## Generation

Every branch reports the size of the local system and the number of non-zero entries. The generated
code is checked for symbolic leftovers before the file is written.

In [ ]:
output_path = generator.Generate(generator.ALM_FRICTIONLESS_COMPONENTS, frictionless_components_functional, TEMPLATE_DIR, OUTPUT_DIR, COMBINATIONS, NORMAL_VARIATIONS)

## Check of the generated file

The file must contain, for each of the geometries and normal-variation flags requested, one
`CalculateLocalLHS` specialisation and one `StaticCalculateLocalRHS` specialisation (a full body for
`false`, a forwarder for `true`).

In [ ]:
import re

with open(output_path) as generated_file:
    generated = generated_file.read()

lhs_specialisations = re.findall(r"^void AugmentedLagrangianMethodFrictionlessComponentsMortarContactCondition<(\d+),\s*(\d+), (true|false), (\d+)>::CalculateLocalLHS\(", generated, re.MULTILINE)
rhs_specialisations = re.findall(r"^void AugmentedLagrangianMethodFrictionlessComponentsMortarContactCondition<(\d+),\s*(\d+), (true|false), (\d+)>::StaticCalculateLocalRHS\(", generated, re.MULTILINE)
expected = len(COMBINATIONS) * len(NORMAL_VARIATIONS)
print("{} lines, {} LHS and {} RHS specialisations (expected {} each)".format(generated.count("\n"), len(lhs_specialisations), len(rhs_specialisations), expected))
assert len(lhs_specialisations) == expected and len(rhs_specialisations) == expected
assert "Derivative(" not in generated and "//subsvar_" not in generated